# RAG System: My Psychologist
**Status:** Personal project / prototype

---

## Why This Project
Many people struggle with psychological challenges but lack accessible support. This system acts as a personal AI psychologist — available anytime — to help users reflect, process emotions, and understand their own patterns.

---

## Problem It Solves
High rates of depression and suicide are partly driven by lack of accessible mental health support. This tool gives users a private, always-on companion that knows their personal context and can guide them through difficult moments.

> Not a replacement for professional therapy — a first layer of support.

---

## What It Does
- Chat naturally with the user about their life, emotions, and experiences
- Pull from the user's personal documents to give context-aware responses
- Spot patterns in the user's thinking (e.g. recurring negativity, triggers)
- Visualize the user's thought/document landscape in 3D

---

## Personal Documents It Uses
Identity, life experiences, beliefs, journal reflections, recent activity, and personal goals — these form the user's knowledge base that the RAG system retrieves from.

---

## Tech Stack
| Tool | Role |
|---|---|
| LangChain | RAG chain, document loading & splitting |
| OpenAI GPT-4o-mini | Chat + embeddings |
| Chroma DB | Vector storage & retrieval |
| Plotly + t-SNE | 3D visualization of embeddings |
| Gradio | Chat UI |

---

## Build Order
1. Document ingestion → embed → store in Chroma
2. RAG chain with conversational memory
3. Gradio chat UI
4. Pattern insight prompts
5. Plotly 3D visualization

NB: Make sure your .env file contains your correct OPENAI_API_KEY

In [ ]:
import os
import glob
from dotenv import load_dotenv
import gradio as gr

from langchain.document_loaders import DirectoryLoader, TextLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from sklearn.manifold import TSNE
import numpy as np
import plotly.graph_objects as go
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
import plotly.express as px


In [ ]:
MODEL = "gpt-4o-mini"
db_name = "personal/documents"

In [ ]:
load_dotenv(override=True)
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# Load and Chunk the user personal documents
1. Scan "personal/documents/" directory for all available documents"
2. Load all markdown file from each category folder
3. Tag each dcument with its category (Identity, Belief, ambission, knowledge, goal, focus, recent activities, purpose, reflections) where possible
4. Split documents into chunks (1000 chars with 200 char overlap)

Why chunk? Large documents are split into smaller pieces for better semantic search precision. Overlap ensures context isn't lost at chunk boundaries.



In [ ]:
folders = glob.glob("personal/documents/*")

def add_metadata(doc, doc_type):
    doc.metadata["doc_type"] = doc_type
    return doc

text_loader_kwargs = {"encoding": "utf-8", "errors": "ignore"}

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs=text_loader_kwargs)
    folder_docs = loader.load()
    documents.extend([add_metadata(doc, doc_type) for doc in folder_docs])

text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)
doc_list = list(set(doc.metadata['doc_type'] for doc in documents))

print(f"Total number of chunks: {len(chunks)}")
print(f"Document types: {doc_list}")    


    

# Create Vector Embeddings and Store in Chroma DB

What happens here:
1. OpenAI Ebeddings converts each text chunk into a 1,536-dimensional vector.
2. Chroma Db stores these embeddings with metadata for fast retrieval.
3. Clean slate - Deletes existings collections if present, then rebuild.

During embeddings: Text is converted to numbers (vectors) that captures semantic meaning, similarities, enabling semantic search.

Vector Dimensions: 1,536 (OpenAI's text-ebedding-ada-002 model)

Storage: Persists to disk so we don't have to re-embed on every execution.


In [ ]:
embeddings = OpenAIEmbeddings()

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vector store created with {vectorstore._collection.count()} documents.")

# Inspect the Vector Store
### verify 
- Total number of vectors stored
- Dimensionality of each vector (Should be 1,536)
- Collection is accessible and queryable

In [ ]:
collection = vectorstore._collection
count = collection.count()

example_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(example_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store.")



# Visualize vector space in 3D

What to expect:
- Each point = one chunk of the document
- colors = document categories if available
- Proximity = semantic similarity (similar topics cluster together)

Why Visualization is a good idea, it reveals the user thoughts processes, beliefs, and knowledge naturally cluster and relate to each other.

In [ ]:
# Get data from the collection first
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Visualization of Document Embeddings',
    scene=dict(
        xaxis_title='TSNE Dimension 1',
        yaxis_title='TSNE Dimension 2',
        zaxis_title='TSNE Dimension 3'
    ),
    width=900,
    height=700,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

# It is time to Build the conversational RAG System

### Components:
- LLM: GPT-4o-mini with temperature 0.7 (Creative but focused responses)
- Memory: ConversationBufferMemory tracks chat history for context
- Retiever: Search vector store for relevant chunks based on questions.
- Chain: ConversationalRetrievalChain orchestrates: query --> retrieve --> generate answer

### How it works:
1. The user ask a question about his/her self.
2. The system search the existing information for a relevant chunks.
3. Retrieve chunks + chat history --> LLM
4. LLM generate contextual response by asking other relevant questions to the user, based on the existing information about the user and then make some recommendations.

# Key Features:
- Warm greetings - Natural first-contact pleasantries, then get down to business.
- emotional Intelligent - Empathy and understanding responses
- No cliches - Avoids "What can I do for you?" type phrases
- Graceful unknowns and refer -Instead of "I don't know", says "what the information I've documented for far...."
- Conversation tone - Speak like a Psychologist would, occasionally using Nigerian expressions.
- Authentic - Real, relatable, not stiff or overly formal.



In [ ]:
# Custom system prompt for emotionally intelligent, empathetic responses, human-like psychologist persona.
from langchain.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate, ChatPromptTemplate

system_prompt = """
You are an experienced and compassionate psychologist with deep expertise in emotional well-being, 
human behavior, and reflective guidance. Your role is to support users by listening carefully, 
understanding their feelings, and responding with empathy, insight, and emotional intelligence.

You communicate in a warm, calm, and human-like manner. You practice active listening, validate 
the user’s emotions, and encourage thoughtful self-reflection. When appropriate, you gently guide 
users toward healthier perspectives, coping strategies, and constructive ways of thinking.

Your responses should:
- Demonstrate empathy and understanding of the user's emotional state.
- Reflect careful listening by acknowledging the user’s feelings and experiences.
- Provide thoughtful insights that help the user reflect on their situation.
- Offer supportive suggestions rather than authoritative commands.
- Encourage self-awareness, resilience, and emotional growth.

You do not judge, criticize, or dismiss the user’s experiences. Instead, you create a safe and 
supportive conversational space where the user feels heard and understood.

If a user expresses distress, sadness, or emotional difficulty, respond with compassion and help 
them explore their thoughts and feelings in a supportive way.

Always prioritize the user's emotional well-being and psychological safety.

Here is what you know about the user:
{context}

Our conversation history is as follows:
{history}
"""

# Create the conversational retrieval chain with the custom system prompt
llm = ChatOpenAI(temperature=0.7, model_name=MODEL)
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True, output_key="answer")
retriever = vectorstore.as_retriever()

# Create custom Prompt
from langchain.chains import ConversationalRetrievalChain

# Combinne documets function
def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs])

conversation_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    return_source_documents=True,
    verbose=True,
    combine_docs_chain_kwargs={
        "prompt": ChatPromptTemplate.from_messages([
            SystemMessagePromptTemplate.from_template(system_prompt),
            HumanMessagePromptTemplate.from_template("{question}")

    ])
    
    }
)

In [ ]:
# Test with example Queries
query = "Hi there!"
result = conversation_chain.invoke({"question": query})
print("Answer:", result["answer"])
print("\n" + "="*50 + "\n")

# Test with actual question
query2 = "What is your name?"
result2 = conversation_chain.invoke({"question": query2})
print("Answer:", result2["answer"])
print("\n" + "="*50 + "\n")




In [ ]:
# Test unknow Handling
query3 = "What is the meaning of engineering?"
result3 = conversation_chain.invoke({"question": query3})
print("Answer:", result3["answer"])
print("\n" + "="*50 + "\n")


In [ ]:
def chat(question, history):
    result = conversation_chain.invoke({"question": question})
    answer = result["answer"]
    return answer #history + [(question, answer)]


# Interactive UI with Gradio

## Feautes:
- Full conversation history maintained
- Warm, emotionally intelligent response
- Opens automatically the browser
- Natural, human-like dialogue flow

Try asking anything about your feelings, emotions, reflections, beliefs, and goals.

In [ ]:
view = gr.ChatInterface(fn=chat, type="messages").launch(inbrowser=True)